# Detección de fraude en tarjetas de crédito

Objetivo: construir un sistema de detección de fraude en tarjetas de crédito usando el dataset etiquetado de construcción.
La idea es intentar que el sistema tenga el mayor AUPR (Área bajo la curva Precision Recall) posible en producción:

from sklearn.metrics import average_precision_score

Se deberá entregar un csv con una única columna con tantas filas como tiene el fichero de producción, y con un score asociado de tal forma que las filas con mayor score sean las que tienen mayor probabilidad de ser fraude.

In [1]:
COLAB = True

In [6]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from matplotlib.ticker import MaxNLocator
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score
from sklearn.metrics import average_precision_score

import matplotlib.pyplot as plt
from IPython.display import clear_output

pd.options.display.max_columns = None

# Carga del dataset

In [3]:
if COLAB:
    aux = "'https://drive.usercontent.google.com/download?id=1IyihCpU_krumRnPBvMR0ytEfzv_Z2Jg3&export=download&confirm=t&uuid=9699f0e2-e760-49fc-b12e-49f140095280'"
    print("wget {} -O ./credit_card_construction.csv.zip".format(aux))
    !wget $aux -O ./credit_card_construction.csv.zip
    !unzip -qq ./credit_card_construction.csv.zip

    aux = "'https://drive.usercontent.google.com/download?id=1N5xRbkchTR7FLskuIEmZPYHklsRVJy_q&export=download&confirm=t&uuid=9699f0e2-e760-49fc-b12e-49f140095280'"
    print("wget {} -O ./credit_card_production.csv.zip".format(aux))
    !wget $aux -O ./credit_card_production.csv.zip
    !unzip -qq ./credit_card_production.csv.zip

wget 'https://drive.usercontent.google.com/download?id=1IyihCpU_krumRnPBvMR0ytEfzv_Z2Jg3&export=download&confirm=t&uuid=9699f0e2-e760-49fc-b12e-49f140095280' -O ./credit_card_construction.csv.zip
--2026-10-15 16:39:04--  https://drive.usercontent.google.com/download?id=1IyihCpU_krumRnPBvMR0ytEfzv_Z2Jg3&export=download&confirm=t&uuid=9699f0e2-e760-49fc-b12e-49f140095280
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.250.145.132, 2a00:1450:4013:c14::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.250.145.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6688969 (6.4M) [application/octet-stream]
Saving to: ‘./credit_card_construction.csv.zip’

./credit_card_const 100%[===================>]   6.38M  --.-KB/s    in 0.06s   

2026-10-15 16:39:05 (98.3 MB/s) - ‘./credit_card_construction.csv.zip’ saved [6688969/6688969]

wget 'https://drive.usercontent.google.com/download?id=1N5xRbkchTR7FLskuIEmZPYHklsR

In [4]:
df_const = pd.read_csv("./credit_card_construction.csv")
df_const

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15,X16,X17,X18,X19,X20,Amount,Class
0,374.0,-288.0,-150.0,-232.0,-90.0,262.0,-262.0,88.0,-26.0,152.0,212.0,60.0,98.0,-50.0,134.0,178.0,64.0,-322.0,-58.0,20.0,144.00,0
1,436.0,-202.0,-566.0,-652.0,166.0,58.0,-18.0,-0.0,54.0,-54.0,214.0,214.0,-2.0,198.0,248.0,-688.0,108.0,100.0,-34.0,-136.0,10.46,0
2,388.0,-128.0,-56.0,30.0,-114.0,104.0,-210.0,72.0,298.0,2.0,28.0,78.0,-168.0,22.0,82.0,172.0,-218.0,222.0,36.0,-40.0,31.72,0
3,-486.0,4.0,314.0,-80.0,-276.0,-94.0,-34.0,94.0,-252.0,2.0,-198.0,-24.0,116.0,8.0,138.0,-110.0,-86.0,326.0,-358.0,-182.0,134.85,0
4,258.0,-238.0,300.0,-70.0,-374.0,122.0,-350.0,92.0,134.0,94.0,4.0,-40.0,-260.0,-192.0,-382.0,182.0,120.0,-120.0,282.0,-6.0,8.70,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170879,-398.0,346.0,-222.0,-534.0,322.0,608.0,-84.0,288.0,190.0,204.0,-50.0,4.0,-0.0,-8.0,152.0,196.0,-216.0,2.0,-116.0,156.0,22.24,0
170880,-88.0,212.0,316.0,-6.0,8.0,-180.0,146.0,-22.0,-102.0,-66.0,-54.0,76.0,168.0,-4.0,152.0,32.0,-88.0,-90.0,2.0,32.0,3.87,0
170881,166.0,-530.0,-632.0,42.0,-112.0,-314.0,322.0,-186.0,-264.0,136.0,-182.0,16.0,126.0,152.0,-70.0,-444.0,38.0,136.0,-182.0,178.0,1122.06,0
170882,-304.0,-1258.0,-528.0,266.0,-334.0,392.0,272.0,16.0,150.0,-140.0,174.0,130.0,-52.0,142.0,258.0,62.0,-40.0,-26.0,-296.0,660.0,2657.25,0


In [8]:
100*df_const["Class"].value_counts()/len(df_const)

,count
Class,
0,99.827368
1,0.172632


In [9]:
df_const.describe()

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15,X16,X17,X18,X19,X20,Amount,Class
count,170884.000000,170884.000000,170884.000000,170884.000000,170884.00000,170884.000000,170884.000000,170884.000000,170884.000000,170884.000000,170884.000000,170884.000000,170884.000000,170884.000000,170884.000000,170884.000000,170884.000000,170884.000000,170884.000000,170884.000000,170884.000000,170884.000000
mean,0.589499,-0.050923,0.285773,-0.097376,0.28397,0.048828,0.005431,0.217773,-0.020575,-0.082852,0.285469,0.235470,-0.286861,-0.195349,-0.006894,0.196554,-0.288933,-0.278130,0.302778,0.080195,132.118705,0.001726
std,390.920605,330.501997,302.081068,282.572168,270.76069,263.927926,241.111627,238.590168,219.366112,216.787580,204.496836,199.591426,198.824463,191.995036,183.039589,175.058560,169.997903,167.325595,162.131421,152.471380,363.489116,0.041513
min,-11282.000000,-14544.000000,-6594.000000,-1136.000000,-6418.00000,-5232.000000,-8302.000000,-14644.000000,-2686.000000,-4880.000000,-868.000000,-3736.000000,-1158.000000,-3764.000000,-840.000000,-2826.000000,-5032.000000,-1900.000000,-1442.000000,-4730.000000,0.000000,0.000000
25%,-184.000000,-120.000000,-178.000000,-170.000000,-138.00000,-154.000000,-112.000000,-42.000000,-128.000000,-106.000000,-152.000000,-82.000000,-130.000000,-86.000000,-116.000000,-94.000000,-96.000000,-100.000000,-90.000000,-42.000000,8.520000,0.000000
50%,4.000000,14.000000,36.000000,-4.000000,-12.00000,-54.000000,8.000000,4.000000,-10.000000,-18.000000,-6.000000,28.000000,-4.000000,10.000000,10.000000,14.000000,-14.000000,0.000000,0.000000,-12.000000,33.000000,0.000000
75%,264.000000,162.000000,206.000000,150.000000,122.00000,80.000000,114.000000,66.000000,120.000000,92.000000,148.000000,124.000000,132.000000,98.000000,130.000000,104.000000,80.000000,100.000000,92.000000,26.000000,115.445000,0.000000
max,490.000000,4294.000000,1876.000000,3344.000000,6960.00000,4278.000000,6860.000000,4002.000000,3118.000000,4750.000000,2334.000000,1570.000000,874.000000,2106.000000,1166.000000,1412.000000,1850.000000,1008.000000,1114.000000,7884.000000,29484.800000,1.000000


In [5]:
df_prod = pd.read_csv("./credit_card_production.csv")
df_prod

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15,X16,X17,X18,X19,X20,Amount,Class
0,234.0,48.0,58.0,106.0,-30.0,-56.0,-22.0,20.0,-38.0,-30.0,360.0,172.0,-32.0,-2.0,136.0,80.0,8.0,-44.0,-52.0,-24.0,1.94,NaN
1,408.0,-126.0,-426.0,-312.0,12.0,-238.0,56.0,-48.0,292.0,-136.0,154.0,118.0,-372.0,204.0,-48.0,-214.0,-4.0,24.0,270.0,-44.0,60.68,NaN
2,-150.0,84.0,580.0,-326.0,-86.0,-100.0,74.0,-32.0,494.0,-418.0,228.0,-348.0,334.0,184.0,30.0,-172.0,142.0,-28.0,-144.0,-18.0,17.77,NaN
3,296.0,-134.0,-388.0,132.0,-12.0,-228.0,114.0,-70.0,44.0,-88.0,304.0,208.0,82.0,-112.0,34.0,78.0,32.0,192.0,-54.0,78.0,416.38,NaN
4,-594.0,-214.0,414.0,308.0,38.0,196.0,-46.0,62.0,70.0,98.0,270.0,32.0,-202.0,-34.0,236.0,-86.0,42.0,28.0,100.0,-222.0,138.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113918,392.0,-98.0,-42.0,78.0,-114.0,42.0,-202.0,36.0,490.0,-38.0,310.0,-406.0,134.0,312.0,-176.0,100.0,24.0,160.0,-42.0,-54.0,7.54,NaN
113919,234.0,16.0,-48.0,146.0,36.0,88.0,-64.0,56.0,98.0,-126.0,136.0,62.0,-172.0,-248.0,-108.0,56.0,180.0,88.0,46.0,-22.0,18.46,NaN
113920,-222.0,106.0,430.0,-176.0,74.0,-58.0,152.0,-26.0,34.0,-118.0,-66.0,108.0,110.0,-142.0,-78.0,-88.0,0.0,-270.0,-74.0,18.0,1.34,NaN
113921,-82.0,380.0,-272.0,182.0,322.0,-46.0,238.0,40.0,-206.0,-176.0,-258.0,18.0,202.0,-198.0,-46.0,-82.0,232.0,94.0,162.0,26.0,23.62,NaN
